# 1. Instalação das Dependências Necessárias

In [1]:
# Instalação das bibliotecas necessárias (incluindo suporte a callbacks e tiktoken)
!pip install openai-whisper librosa deep-translator transformers langchain-core langchain-openai langchain-community tiktoken

# 2. Configuração e Importação de Bibliotecas

In [2]:
import os
import getpass
import librosa
import numpy as np
import whisper
from deep_translator import GoogleTranslator
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
# Importação do Callback do LangChain para monitoramento estrito de tokens
from langchain_community.callbacks import get_openai_callback

# Configuração da chave de API
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Insira sua OpenAI API Key: ")

print("✅ Ambiente e ferramentas de auditoria configurados com sucesso!")

/tmp/ipykernel_7036/3726610662.py:10: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks import get_openai_callback


Insira sua OpenAI API Key: ··········
✅ Ambiente e ferramentas de auditoria configurados com sucesso!


# 3. Motor de Análise Vocal (Tom de Voz e Hesitação)

In [3]:
def analisar_tom_e_hesitacao(audio_path):
    print("🧠 Etapa 2: Analisando tom de voz e hesitações do paciente localmente...")
    y, sr = librosa.load(audio_path, sr=None)

    # Detecção de silêncios / Hesitações
    intervals = librosa.effects.split(y, top_db=30)
    duracao_total = librosa.get_duration(y=y, sr=sr)
    tempo_ativo = sum([(end - start) / sr for start, end in intervals])
    tempo_silencio = duracao_total - tempo_ativo
    taxa_hesitacao = (tempo_silencio / duracao_total) * 100 if duracao_total > 0 else 0

    # Análise de Pitch (F0)
    pitches, magnitudes = librosa.piptrack(y=y, sr=sr)
    pitch_valores = pitches[pitches > 0]
    variacao_pitch = np.std(pitch_valores) if len(pitch_valores) > 0 else 0

    status_vocal = "Estável"
    if taxa_hesitacao > 25:
        status_vocal = "Alta Hesitação (Paciente pensativo ou inseguro ao relatar sintomas)"
    elif variacao_pitch > 50:
        status_vocal = "Tom de voz instável/Emocional (Sinais potenciais de ansiedade ou dor)"

    return {
        "taxa_hesitacao_porcento": round(taxa_hesitacao, 2),
        "variacao_tom": round(variacao_pitch, 2),
        "diagnostico_vocal": status_vocal
    }

# 4. Motor de Transcrição (Whisper em inglês)

In [4]:
def transcrever_audio_en(audio_path):
    print("\n🎙️ Etapa 3: Transcrevendo o áudio original com OpenAI Whisper...")
    model = whisper.load_model("base")

    # O Whisper processa localmente o áudio em inglês sem enviar tradução prévia à LLM (para economizar tokens)
    result = model.transcribe(audio_path, language="en")
    transcricao_ingles = result["text"]
    print(f"📝 Transcrição bruta em Inglês concluída (Economia de caracteres mantida).")
    return transcricao_ingles

# 5. Inteligência Generativa (Mapeamento Clínico e Prontuário SOAP)

In [5]:
def gerar_prontuario_soap_en(texto_en, insights_audio):
    print("\n🏥 Etapa 4: Configurando os parâmetros da LLM (Teto rígido e Prompt Defensivo)...")

    # MODIFICAÇÃO: Forçando um teto rígido definindo o parâmetro max_tokens na inicialização do modelo
    llm = ChatOpenAI(
        model="gpt-4o",
        temperature=0.2,
        max_tokens=600  # Impede respostas prolixas ou alucinações longas que gastam tokens
    )

    # Prompt escrito em inglês com restrições explícitas de tamanho (Engenharia de Prompt Defensiva para evitar comportamentos indesejados)
    template_clinico = """
    You are an Expert Medical AI Auditor. Analyze the provided English consultation transcript and patient acoustic metrics.
    Structure a highly concise medical record under the SOAP framework.

    Constraints:
    - Be extremely brief and direct. Use short bullet points only. Do not write full paragraphs.
    - If a section has no data, simply write "Not reported". Do not make up explanations.

    Input Data:
    - Transcript: {transcricao}
    - Acoustic Status: {analise_vocal}

    SOAP Sections required:
    - S (Subjective): Patient complaints, symptoms, and vocal hesitation/stress insights.
    - O (Objective): Measurable data, vitals or exams if mentioned.
    - A (Assessment): Clinical hypothesis or severity.
    - P (Plan): Medications, further testing or follow-up.

    Output the report in crisp Markdown format.
    """

    prompt = PromptTemplate(
        input_variables=["transcricao", "analise_vocal"],
        template=template_clinico
    )

    chain = prompt | llm

    payload_analise = f"Status: {insights_audio['diagnostico_vocal']} | Hesitation: {insights_audio['taxa_hesitacao_porcento']}%"
    resposta_prontuario = chain.invoke({"transcricao": texto_en, "analise_vocal": payload_analise})

    return resposta_prontuario.content

# 6. Execução da Pipeline Completa com Tradução Local Final

In [6]:
# --- CARREGAMENTO DO ARQUIVO LOCALMENTE (SESSÃO DO GOOGLE COLAB) ---
audio_clinico = "/content/CAR0001.mp3"

# 1. Extração de Áudio Local
insights_vocal = analisar_tom_e_hesitacao(audio_clinico)

# 2. Transcrição Local em Inglês
texto_en = transcrever_audio_en(audio_clinico)

# MODIFICAÇÃO: Encapsulando a chamada com o get_openai_callback para auditoria financeira
print("\n⚡ Enviando requisição monitorada para a API da OpenAI...")
with get_openai_callback() as cb:
    # A LLM recebe dados em inglês e responde em inglês de forma enxuta
    prontuario_en = gerar_prontuario_soap_en(texto_en, insights_vocal)

    # Exibição detalhada do consumo gerado na operação
    print("\n" + "="*45)
    print("📊 AUDITORIA DO CONSUMO DE TOKENS (FLUXO OTIMIZADO)")
    print(f"Tokens de Entrada (Prompt): {cb.prompt_tokens}")
    print(f"Tokens de Saída (Resposta): {cb.completion_tokens}")
    print(f"Total de Tokens Utilizados: {cb.total_tokens}")
    print(f"Custo Real da Operação: ${cb.total_cost:.5f}")
    print("="*45)

# 3. Tradução Externa Local Gratuita (Trocando a ordem da tradução para o final)
print("\n🌍 Etapa 5: Traduzindo o Prontuário Final SOAP para o Português (PT-BR)...")
tradutor = GoogleTranslator(source='en', target='pt')
prontuario_final_pt = tradutor.translate(prontuario_en)

print("\n" + "="*50 + "\n📄 OUTPUT FINAL: PRONTUÁRIO MÉDICO TRADUZIDO (SOAP)\n" + "="*50)
print(prontuario_final_pt)

🧠 Etapa 2: Analisando tom de voz e hesitações do paciente localmente...

🎙️ Etapa 3: Transcrevendo o áudio original com OpenAI Whisper...


100%|████████████████████████████████████████| 139M/139M [00:01<00:00, 138MiB/s]
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


📝 Transcrição bruta em Inglês concluída (Economia de caracteres mantida).

⚡ Enviando requisição monitorada para a API da OpenAI...

🏥 Etapa 4: Configurando os parâmetros da LLM (Teto rígido e Prompt Defensivo)...

📊 AUDITORIA DO CONSUMO DE TOKENS (FLUXO OTIMIZADO)
Tokens de Entrada (Prompt): 1488
Tokens de Saída (Resposta): 185
Total de Tokens Utilizados: 1673
Custo Real da Operação: $0.00557

🌍 Etapa 5: Traduzindo o Prontuário Final SOAP para o Português (PT-BR)...

📄 OUTPUT FINAL: PRONTUÁRIO MÉDICO TRADUZIDO (SOAP)
```redução
# Registro Médico

## S (Subjetivo)
- Queixa: Dor no peito
- Início: Ontem à noite, constante, agudo
- Localização: lado esquerdo do peito
- Gravidade: 7-8/10
- Fatores agravantes: Deitar, respirar fundo
- Sintomas associados: tontura, dificuldade para respirar, leve aceleração cardíaca, inchaço no pescoço
- Fumar: 1 maço/dia por 10-15 anos
- História familiar: o pai teve um ataque cardíaco aos 45 anos
- Hesitação vocal: 32,5% (Alta Hesitação)

## O (Objetivo)
